# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [8]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked,
    convert_to_proper_CRS_and_cogify_ultra_large
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [9]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [10]:

EVENT_NAME = '202409_Hurricane_Helene'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'aria'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [11]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [12]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 8 .tif files in the S3 bucket.


['drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/OPERA_DSWx-S1_BWTR_ChngMap_20240926-20240914_v2.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/RTC/OPERA_RTC_S1_20240926_mosaic_RGB_compressed.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [13]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [14]:
# Check current cache status using the imported function
check_cache_status()

📁 Cache directory does not exist: data_download/
   Creating cache directory...
✅ Cache directory created: data_download/


(0, 0)

In [15]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [16]:
keys

['drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/OPERA_DSWx-S1_BWTR_ChngMap_20240926-20240914_v2.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/RTC/OPERA_RTC_S1_20240926_mosaic_RGB_compressed.tif']

In [19]:
# Define filename creator functions for different file types

def create_cog_filename_distAlert(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    
    # Extract the date from the end of the filename (last 8 characters)
    date_str = f2[-8:]  # Gets '20241002'
    
    # Get everything except the date
    prefix = f2[:-8]  # Gets 'ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_'
    
    # Convert YYYYMMDD to YYYY-MM-DD format
    formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
    
    # Create filename: EVENT_NAME + prefix + formatted date + _day.tif
    cog_filename = f'{EVENT_NAME}_{prefix}{formatted_date}_day.tif'
    return cog_filename


filter_str = 'OPERA-DIST-ALERT'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_distAlert(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-02_day.tif
  202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_2024-10-02_day.tif


In [20]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_distAlert, 
                                target_dir = "Sentinel-2/distAlert", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-02_day.tif
  202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_2024-10-02_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202409_Hurricane_Helene/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-2/distAlert

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202409_Hurricane_Helene

[1/2] Processing: drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241002.tif
   Output filename: 202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-02_day.tif
   [MEMORY] Initial: 301.0 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata value: 255.0
   [CHUNKS

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=99, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...


Reading input: /tmp/tmp4ttg2zl4_temp.tif



   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpy6neajk_.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-02_day.tif
   [MEMORY] Final: 442.8 MB (Change: +141.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-02_day.tif

[2/2] Processing: drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_20241002.tif
   Output filename: 202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_2024-10-02_day.tif
   [MEMORY] Initial: 442.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
  

Reading input: /tmp/tmp5fndogfq_temp.tif                   



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=8, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpo42a8awd.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_2024-10-02_day.tif
   [MEMORY] Final: 471.6 MB (Change: +28.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_2024-10-02_day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025

In [30]:
keys

['drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/OPERA_DSWx-S1_BWTR_ChngMap_20240926-20240914_v2.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/RTC/OPERA_RTC_S1_20240926_mosaic_RGB_compressed.tif']

In [22]:
# Define filename creator functions for different file types

def create_cog_filename_dsw(f, EVENT_NAME):
    """Create COG filename for DSWx files with date formatting."""
    f2 = Path(f).stem
    
    # Extract date and file type from filename
    # Format is: YYYYMMDD_DSWx-S1_TYPE
    parts = f2.split('_')
    
    if len(parts) >= 2:
        date_str = parts[0]  # '20240926'
        file_type = '_'.join(parts[1:])  # 'DSWx-S1_WTR' or 'DSWx-S1_BWTR'
        
        # Convert date from YYYYMMDD to YYYY-MM-DD
        if len(date_str) == 8 and date_str.isdigit():
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        else:
            formatted_date = date_str  # Keep original if not in expected format
        
        # Create new filename: EVENT_NAME_TYPE_DATE.tif
        cog_filename = f'{EVENT_NAME}_{file_type}_{formatted_date}_day.tif'
    else:
        # Fallback if format is unexpected
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename


filter_str = 'DSWx/2024'  # This will catch files starting with dates in the DSWx folder

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dsw(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202409_Hurricane_Helene_DSWx-S1_BWTR_2024-09-14_day.tif
  202409_Hurricane_Helene_DSWx-S1_WTR_2024-09-14_day.tif
  202409_Hurricane_Helene_DSWx-S1_BWTR_2024-09-26_day.tif
  202409_Hurricane_Helene_DSWx-S1_WTR_2024-09-26_day.tif


In [23]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_dsw, 
                                target_dir = "Sentinel-1/opera_dswx", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_DSWx-S1_BWTR_2024-09-14_day.tif
  202409_Hurricane_Helene_DSWx-S1_WTR_2024-09-14_day.tif
  202409_Hurricane_Helene_DSWx-S1_BWTR_2024-09-26_day.tif
  202409_Hurricane_Helene_DSWx-S1_WTR_2024-09-26_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202409_Hurricane_Helene/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/opera_dswx

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202409_Hurricane_Helene

[1/4] Processing: drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_BWTR.tif
   Output filename: 202409_Hurricane_Helene_DSWx-S1_BWTR_2024-09-14_day.tif
   [MEMORY] Initial: 471.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata value: 255.0
  

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=1, max=1, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpovvjvoz0_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp9ga4kwae.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=3, center sample non-zero=1000000/1000000
            Estimated data coverage: 40.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpsk4xutwu_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpz05hll6o.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/202409_Hurricane_Helene_DSWx-S1_WTR_2024-09-26_day.tif
   [MEMORY] Final: 620.8 MB (Change: +5.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_DSWx-S1_WTR_2024-09-26_day.tif

✅ Batch processing complete: 4 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 4
Successful: 4
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T00:23:17.563481


In [24]:
keys

['drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/OPERA_DSWx-S1_BWTR_ChngMap_20240926-20240914_v2.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/RTC/OPERA_RTC_S1_20240926_mosaic_RGB_compressed.tif']

In [27]:
# Define filename creator functions for different file types

def create_cog_filename_chngMap(f, EVENT_NAME):
    """Create COG filename for water mask files, reversing dates for BWTR_ChngMap files."""
    f2 = Path(f).stem
    
    # Check if it's a BWTR_ChngMap file with dates in format YYYYMMDD-YYYYMMDD
    if 'BWTR_ChngMap' in f2 and '-' in f2:
        # Split to get the date part
        parts = f2.split('_')
        
        # Find where ChngMap is and check if there's a version after it
        chngmap_index = None
        version_suffix = ""
        
        for i, part in enumerate(parts):
            if part == 'ChngMap':
                chngmap_index = i
                # Check if next part after dates is a version
                if i + 2 < len(parts) and parts[i + 2].startswith('v'):
                    version_suffix = parts[i + 2]
            elif '-' in part and len(part) == 17:  # YYYYMMDD-YYYYMMDD
                date1, date2 = part.split('-')
                
                # Format dates as YYYY-MM-DD
                formatted_date1 = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}"
                formatted_date2 = f"{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
                
                # Reverse the dates and format with 'c' prefix
                parts[i] = f'c{formatted_date2}_{formatted_date1}'
        
        # Reconstruct: put version right after ChngMap if it exists
        if version_suffix and chngmap_index is not None:
            # Remove version from its current position
            parts = [p for p in parts if p != version_suffix]
            # Insert version right after ChngMap
            parts.insert(chngmap_index + 1, version_suffix)
        
        f2 = '_'.join(parts)
    
    cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    return cog_filename

filter_str = 'BWTR_ChngMap'  # This will catch files starting with dates in the DSWx folder

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_chngMap(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202409_Hurricane_Helene_OPERA_DSWx-S1_BWTR_ChngMap_v2_c2024-09-14_2024-09-26_day.tif


In [28]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_chngMap, 
                                target_dir = "Sentinel-1/opera_dswx", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_OPERA_DSWx-S1_BWTR_ChngMap_v2_c2024-09-14_2024-09-26_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202409_Hurricane_Helene/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/opera_dswx

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202409_Hurricane_Helene

[1/1] Processing: drcs_activations/202409_Hurricane_Helene/aria/DSWx/OPERA_DSWx-S1_BWTR_ChngMap_20240926-20240914_v2.tif
   Output filename: 202409_Hurricane_Helene_OPERA_DSWx-S1_BWTR_ChngMap_v2_c2024-09-14_2024-09-26_day.tif
   [MEMORY] Initial: 620.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 4.00 MB
   [NODATA] Source nodata value: -3.4028234663852886e+38
   [CHUNKS] Processing 437 chunks (19x23)
   [BAND 1/1] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.0, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpgeph9r7g_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpahmo6mib.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/202409_Hurricane_Helene_OPERA_DSWx-S1_BWTR_ChngMap_v2_c2024-09-14_2024-09-26_day.tif
   [MEMORY] Final: 756.7 MB (Change: +135.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_OPERA_DSWx-S1_BWTR_ChngMap_v2_c2024-09-14_2024-09-26_day.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 

In [29]:
keys

['drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DIST/ARIA_OPERA-DIST-ALERT_VEG-DIST-STATUS_S2A_20241002.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240914_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_BWTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/20240926_DSWx-S1_WTR.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/DSWx/OPERA_DSWx-S1_BWTR_ChngMap_20240926-20240914_v2.tif',
 'drcs_activations/202409_Hurricane_Helene/aria/RTC/OPERA_RTC_S1_20240926_mosaic_RGB_compressed.tif']

In [31]:
from pathlib import Path

def create_cog_filename_RTC(f, EVENT_NAME):
    """Create COG filename for RTC files, moving date to the end."""
    f2 = Path(f).stem
    
    # Split by underscore
    parts = f2.split('_')
    
    # Find the date (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Remove date from its current position
        parts.pop(date_index)
        
        # Convert date from YYYYMMDD to YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Reconstruct filename with date at the end
        f2 = '_'.join(parts) + f'_{formatted_date}'
    
    cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    return cog_filename


filter_str = 'OPERA_RTC'
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_RTC(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202409_Hurricane_Helene_OPERA_RTC_S1_mosaic_RGB_compressed_2024-09-26_day.tif


In [32]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_RTC, 
                                target_dir = "Sentinel-1/rgb", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_OPERA_RTC_S1_mosaic_RGB_compressed_2024-09-26_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202409_Hurricane_Helene/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/rgb

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202409_Hurricane_Helene

[1/1] Processing: drcs_activations/202409_Hurricane_Helene/aria/RTC/OPERA_RTC_S1_20240926_mosaic_RGB_compressed.tif
   Output filename: 202409_Hurricane_Helene_OPERA_RTC_S1_mosaic_RGB_compressed_2024-09-26_day.tif
   [MEMORY] Initial: 757.0 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 4.00 MB
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [NODATA] Source nodata value: 0
   [CHUNKS] Processing 322 chunks (14x23)
 

   [BAND 2/4] Processing...


   [BAND 3/4] Processing...


   [BAND 4/4] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=18, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [VERIFY] Band 2: min=2, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [VERIFY] Band 3: min=0, max=255, center sample non-zero=999911/1000000
            Estimated data coverage: 79.3% (from distributed samples)
   [VERIFY] Band 4: min=255, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmpgm1_nq9c_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp4f8mrcvk.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/202409_Hurricane_Helene_OPERA_RTC_S1_mosaic_RGB_compressed_2024-09-26_day.tif
   [MEMORY] Final: 839.2 MB (Change: +82.2 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_OPERA_RTC_S1_mosaic_RGB_compressed_2024-09-26_day.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T00:31:18.393846


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")